In [1]:
import pandas as pd
import zipfile
import json
import numpy as np
import jsonlines
import random

from schema_functions import create_schema, abbreviate_schema

Set file paths - update drive_loc as required

In [ ]:
# Set paths
drive_loc = '***' ## *** Update to match your own system
output_dir = 'sql_data'

## File paths - these should match the folder structure in the zip files
roots = {'train': f'{drive_loc}/BIRD/train.zip',
         'dev': f'{drive_loc}/BIRD/dev.zip'}

sql_files = {'train': 'train/train_gold.sql',
             'dev': 'dev_20240627/dev.sql'}

tables = {'train': 'train/train_tables.json',
          'dev': 'dev_20240627/dev_tables.json'}

questions = {'train': 'train/train.json',  
             'dev': 'dev_20240627/dev.json'}

Load all relevant data

In [3]:
# Initialise data stores
queries = []
datasets = []
sources = []
tables_dict = {}
questions_df = pd.DataFrame()

# Load queries
for s in ['train', 'dev']:

    z = zipfile.ZipFile(roots[s])

    # Load tables
    content = ''
    for line in z.open(tables[s]):
        content += line.decode("utf-8")
    tables_dict[s] = json.loads(content)

    # Load SQL queries
    for line in z.open(sql_files[s]):

        #print(line)
        query, dataset = line.decode("utf-8").split('\t')
        queries += [query]
        datasets += [dataset.replace('\n', '')]
        sources += [s]

    # Load questions
    content = ''
    for line in z.open(questions[s]):
        content += line.decode("utf-8")
    questions_df_temp = pd.DataFrame(json.loads(content))
    questions_df_temp['source'] = s
    questions_df = pd.concat([questions_df, questions_df_temp])

Check basic table information

In [4]:
print(f'Total train tables {len(tables_dict['train'])}') ## 69
print(f'Total dev tables {len(tables_dict['dev'])}') ## 11

Total train tables 69
Total dev tables 11


Create schemas

In [7]:
schema_train = create_schema(tables_dict['train'])
schema_dev = create_schema(tables_dict['dev'])

Abreviate schemas to minimise token usage when finetuning

In [8]:
schema_train_abvr = abbreviate_schema(schema_train)
schema_dev_abvr = abbreviate_schema(schema_dev)

Remove schemas with more than 1200 tokens to avoid truncation during fine tuning (will fine tune with 2048 tokens)

In [9]:
schema_train_abvr = {k:v for k, v in schema_train_abvr.items() if len(str(v))/4 < 1200}
schema_dev_abvr = {k:v for k, v in schema_dev_abvr.items() if len(str(v))/4 < 1200}

In [ ]:
# Inspect data
print(schema_train_abvr)

{'european_football_1': {'divisions': {'columns': {'division': 'text',
    'name': 'text',
    'country': 'text'},
   'primary_keys': ['division'],
   'foreign_keys': {'division': {'matchs': 'Div'}}},
  'matchs': {'columns': {'Div': 'text',
    'Date': 'date',
    'HomeTeam': 'text',
    'AwayTeam': 'text',
    'FTHG': 'integer',
    'FTAG': 'integer',
    'FTR': 'text',
    'season': 'integer'},
   'foreign_keys': {'Div': {'divisions': 'division'}}}},
 'sales_in_weather': {'sales_in_weather': {'columns': {'date': 'date',
    'store_nbr': 'integer',
    'item_nbr': 'integer',
    'units': 'integer'},
   'primary_keys': ['date', 'store_nbr', 'item_nbr'],
   'foreign_keys': {'store_nbr': {'relation': 'store_nbr'}}},
  'weather': {'columns': {'station_nbr': 'integer',
    'date': 'date',
    'tmax': 'integer',
    'tmin': 'integer',
    'tavg': 'integer',
    'depart': 'integer',
    'dewpoint': 'integer',
    'wetbulb': 'integer',
    'heat': 'integer',
    'cool': 'integer',
    'sunris

Check statistics

In [13]:
## Check statistics
print(f'Train tables count: {len(schema_train_abvr)}') # 66
print(f'Dev tables count: {len(schema_dev_abvr)}') # 10

## Average tokens per schema reduction - train
print(f'Train tokens per schema before abbreviation: {len(str(schema_train))/(4*len(schema_train))}')  # 957
print(f'Train tokens per schema after abbreviation: {len(str(schema_train_abvr))/(4*len(schema_train))}')  # 393

## Average tokens per schema reduction - dev
print(f'Dev tokens per schema before abbreviation: {len(str(schema_dev))/(4*len(schema_dev))}')  # 1261
print(f'Dev tokens per schema after abbreviation: {len(str(schema_dev_abvr))/(4*len(schema_dev))}') # 494

Train tables count: 66
Dev tables count: 10
Train tokens per schema before abbreviation: 957.7898550724638
Train tokens per schema after abbreviation: 393.6847826086956
Dev tokens per schema before abbreviation: 1261.3863636363637
Dev tokens per schema after abbreviation: 494.72727272727275


Assign some of the train data to the test set

In [ ]:
schema_names_train = list(schema_train.keys())
schema_names_dev = list(schema_dev.keys())

random.seed(1)

## Create test set via random sample
test_schemas = random.sample(schema_names_train, int(len(schema_names_train)*0.2))
questions_df.loc[questions_df['db_id'].isin(test_schemas), 'source'] = 'test'

Compile datasets

In [29]:
## Abreviated schemas
train_data = [{'schema': str(schema_train_abvr[row['db_id']]), 'question': row['question'],  'sql_query': row['SQL']} for _, row in questions_df.iterrows() if row['source'] == 'train' and row['db_id'] in schema_train_abvr.keys()]
valid_data = [{'schema': str(schema_dev_abvr[row['db_id']]), 'question': row['question'],  'sql_query': row['SQL']} for _, row in questions_df.iterrows() if row['source'] == 'dev' and row['db_id'] in schema_dev_abvr.keys()]
test_data = [{'schema': str(schema_train_abvr[row['db_id']]), 'question': row['question'],  'sql_query': row['SQL']} for _, row in questions_df.iterrows() if row['source'] == 'test' and row['db_id'] in schema_train_abvr.keys()]


Check that all questions/schemas have been assigned to the train, validation or test set

In [30]:
conditions = [questions_df['db_id'].isin(schema_names_train) & (questions_df['source'] == 'train'),
              questions_df['db_id'].isin(schema_names_dev) & (questions_df['source'] == 'dev'),
              questions_df['db_id'].isin(schema_names_train) & (questions_df['source'] == 'test'),
              ~questions_df['db_id'].isin(schema_names_train) & ~questions_df['db_id'].isin(schema_names_dev)]
choices = [True, True, True, False]

questions_df['schema_found'] = np.select(conditions, choices, "ERROR")
questions_df['schema_found'].value_counts()

schema_found
True    10962
Name: count, dtype: int64

Check schema statistics

In [ ]:
print(f'Average train tokens: {np.mean([len(str(t))/4 for t in train_data])}') # 600 tokens
print(f'Average validation tokens: {np.mean([len(str(t))/4 for t in valid_data])}')  # 645 tokens
print(f'Average test tokens: {np.mean([len(str(t))/4 for t in test_data])}')  # 540 tokens

print(f'Maximum train tokens: {max([len(str(t))/4 for t in train_data])}') # 1342 tokens
print(f'Maximum validation tokens: {max([len(str(t))/4 for t in valid_data])}') # 1402 tokens
print(f'Maximum test tokens: {max([len(str(t))/4 for t in test_data])}') # 903 tokens


Average train tokens: 592.5658880564125
Average validation tokens: 646.2759786476869
Average test tokens: 532.1486325802616
Maximum train tokens: 1343.0
Maximum validation tokens: 1403.25
Maximum test tokens: 1309.25


Save datasets back to disk

In [ ]:
## Save data
data_dict = {'train': train_data,
             'valid': valid_data,
             'test': test_data}

dataset = 'bird'
for s in ['train', 'valid', 'test']:
    with jsonlines.open(f'{output_dir}/{dataset}_{s}_abrev.jsonl', mode='w') as writer:
        writer.write(data_dict[s])

Upload datasets to AWS

In [ ]:
from sagemaker.s3 import S3Uploader
import sagemaker
import os
import boto3
from botocore.config import Config

# Load profile
aws_profile = os.getenv('aws_profile')
account_id = aws_profile.split('-')[1]
print('aws sso login --profile ' + aws_profile) # Run this command in a terminal and follow instructions to set up AWS connection

# Set up session
boto3_session = boto3.session.Session(profile_name = aws_profile)
session = sagemaker.Session(boto3_session)

In [ ]:
# Get list of files to upload
files = os.listdir(output_dir)

## File upload
bucket_name = 'bucket_name' # ** set bucket name as desired
train_data_location = f"s3://{bucket_name}/data_files"

## Upload all files
for f in files:
    local_data_file = f'{folder}{f}'
    S3Uploader.upload(local_data_file, train_data_location)

    print(f"Training data: {train_data_location}")